In [1]:
from typing import Literal

from pydantic import BaseModel, Field
from typing_extensions import Annotated

import autogen
from autogen.cache import Cache

from dotenv import load_dotenv
import os

load_dotenv()

config_list = [
    {
        'model': 'gpt-3.5-turbo',
        'api_key': os.getenv("OPENAI_API_KEY"),
        'tags': ['tool', '3.5-tool'],
    }
]

In [6]:
llm_config = {
    "config_list": config_list,
    "timeout": 120,
}

chatbot = autogen.AssistantAgent(
    name="chatbot",
    system_message="For currency exchange tasks, only use the functions you have been provided with. Reply TERMINATE when the task is done.",
    llm_config=llm_config,
)

# create a UserProxyAgent instance named "user_proxy"
user_proxy = autogen.UserProxyAgent(
    name="user_proxy",
    is_termination_msg=lambda x: x.get("content", "") and x.get("content", "").rstrip().endswith("TERMINATE"),
    human_input_mode="NEVER",
    max_consecutive_auto_reply=10,
)


class Currency(BaseModel):
    currency: Annotated[CurrencySymbol, Field(..., description="Currency symbol")]
    amount: Annotated[float, Field(0, description="Amount of currency", ge=0)]


# another way to register a function is to use register_function instead of register_for_execution and register_for_llm decorators
def currency_calculator(
    base: Annotated[Currency, "Base currency: amount and currency symbol"],
    quote_currency: Annotated[CurrencySymbol, "Quote currency symbol"] = "USD",
) -> Currency:
    quote_amount = exchange_rate(base.currency, quote_currency) * base.amount
    return Currency(amount=quote_amount, currency=quote_currency)


autogen.agentchat.register_function(
    currency_calculator,
    caller=chatbot,
    executor=user_proxy,
    description="Currency exchange calculator.",
)

In [7]:
chatbot.llm_config["tools"]

[{'type': 'function',
  'function': {'description': 'Currency exchange calculator.',
   'name': 'currency_calculator',
   'parameters': {'type': 'object',
    'properties': {'base': {'properties': {'currency': {'description': 'Currency symbol',
        'enum': ['USD', 'EUR'],
        'title': 'Currency',
        'type': 'string'},
       'amount': {'default': 0,
        'description': 'Amount of currency',
        'minimum': 0.0,
        'title': 'Amount',
        'type': 'number'}},
      'required': ['currency'],
      'title': 'Currency',
      'type': 'object',
      'description': 'Base currency: amount and currency symbol'},
     'quote_currency': {'enum': ['USD', 'EUR'],
      'type': 'string',
      'default': 'USD',
      'description': 'Quote currency symbol'}},
    'required': ['base']}}}]

In [8]:
assert user_proxy.function_map["currency_calculator"]._origin == currency_calculator

In [9]:
with Cache.disk() as cache:
    # start the conversation
    res = user_proxy.initiate_chat(
        chatbot, message="How much is 123.45 USD in EUR?", summary_method="reflection_with_llm", cache=cache
    )

user_proxy (to chatbot):

How much is 123.45 USD in EUR?

--------------------------------------------------------------------------------
chatbot (to user_proxy):

***** Suggested tool call (call_yas7pBBplVI87ORCmLLrMILU): currency_calculator *****
Arguments: 
{"base":{"currency":"USD","amount":123.45},"quote_currency":"EUR"}
************************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING FUNCTION currency_calculator...
user_proxy (to chatbot):

user_proxy (to chatbot):

***** Response from calling tool (call_yas7pBBplVI87ORCmLLrMILU) *****
{"currency":"EUR","amount":112.22727272727272}
**********************************************************************

--------------------------------------------------------------------------------
chatbot (to user_proxy):

123.45 USD is equivalent to 112.23 EUR. TERMINATE

----------------------------------------------------

In [11]:
print("Chat summary:", res.summary)


Chat summary: 123.45 USD is equivalent to 112.23 EUR.


In [12]:
with Cache.disk() as cache:
    # start the conversation
    res = user_proxy.initiate_chat(
        chatbot,
        message="How much is 123.45 US Dollars in Euros?",
        cache=cache,
    )

user_proxy (to chatbot):

How much is 123.45 US Dollars in Euros?

--------------------------------------------------------------------------------
chatbot (to user_proxy):

***** Suggested tool call (call_ElvDKs9Gc37QcmbRCxmXAAP4): currency_calculator *****
Arguments: 
{"base":{"currency":"USD","amount":123.45},"quote_currency":"EUR"}
************************************************************************************

--------------------------------------------------------------------------------

>>>>>>>> EXECUTING FUNCTION currency_calculator...
user_proxy (to chatbot):

user_proxy (to chatbot):

***** Response from calling tool (call_ElvDKs9Gc37QcmbRCxmXAAP4) *****
{"currency":"EUR","amount":112.22727272727272}
**********************************************************************

--------------------------------------------------------------------------------
chatbot (to user_proxy):

123.45 US Dollars is equivalent to 112.23 Euros.

--------------------------------------------

In [14]:
print("Chat history:", res.chat_history)


Chat history: [{'content': 'How much is 123.45 US Dollars in Euros?', 'role': 'assistant', 'name': 'user_proxy'}, {'tool_calls': [{'id': 'call_ElvDKs9Gc37QcmbRCxmXAAP4', 'function': {'arguments': '{"base":{"currency":"USD","amount":123.45},"quote_currency":"EUR"}', 'name': 'currency_calculator'}, 'type': 'function'}], 'content': None, 'role': 'assistant'}, {'content': '{"currency":"EUR","amount":112.22727272727272}', 'tool_responses': [{'tool_call_id': 'call_ElvDKs9Gc37QcmbRCxmXAAP4', 'role': 'tool', 'content': '{"currency":"EUR","amount":112.22727272727272}'}], 'role': 'tool', 'name': 'user_proxy'}, {'content': '123.45 US Dollars is equivalent to 112.23 Euros.', 'role': 'user', 'name': 'chatbot'}, {'content': '', 'role': 'assistant', 'name': 'user_proxy'}, {'content': 'Terminate\nTERMINATE', 'role': 'user', 'name': 'chatbot'}]
